# Program for training simple model for back-classification

Create kernel: `uv run ipython kernel install --user --name="python-classification"`
Switch to this kernel (see top right in jupyter)

Also ran one time: `mlflow experiments create --experiment-name "back_classification" --artifact-location s3://projet-aiml4os-wp10/Cluster5/mlflow_artifacts`
Not sure if I need to now as have changed things around

In [ ]:
import pandas as pd
import os

from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.calibration import CalibratedClassifierCV

import nltk
from nltk.corpus import stopwords

from utils.one_to_one import get_1to1_dict, get_problem_NACE
from utils.rules_classification import norwegian_rules
from utils.preprocess import preprocess

import mlflow
import mlflow.sklearn

In [ ]:
# Location for model and data
bucket = "projet-aiml4os-wp10/Cluster5"
bucket_url=f"https://minio.lab.sspcloud.fr/{bucket}"
train_path = f"{bucket_url}/train_doublenace_2026-03-23.parquet"
conversion_path = f"{bucket_url}/NACE2.1-NACE2_Table_V1.05.xlsx"

In [ ]:
# Set MLflow experiment for ML model if not set from before
os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.environ["S3_ENDPOINT"]

if not mlflow.get_experiment_by_name("back_classification"):
    mlflow.create_experiment(
        "back_classification",
        artifact_location=f"s3://{bucket}/mlflow_artifacts/"
    )

mlflow.set_experiment("back_classification")

### Read in data

In [ ]:
# Read in training data
train = pd.read_parquet(train_path)
print(f'Number of observations in training: {train.shape[0]}')

### Remove 1-to-1 groups

In [ ]:
one_to_one_dict = get_1to1_dict(conversion_path)
mask_1 = train.nace_21_code.isin(one_to_one_dict.keys())
print(f'Number of observations in one-to-one nace: {mask_1.sum()}')

train = train.loc[~mask_1, :].copy()

### Remove problem groups

In [ ]:
problem_nace = get_problem_NACE(conversion_path)
mask_problem = train.nace_21_code.isin(problem_nace)
print(f'Number of observations in problem nace: {mask_problem.sum()}')

train = train.loc[~mask_problem, :].copy()

### Remove rules groups

In [ ]:
mask_rules = train.nace_21_code.isin(norwegian_rules.keys())
print(f'Number of observations in rules nace: {mask_rules.sum()}')

train = train.loc[~mask_rules, :].copy()

In [ ]:
# Remove groups with too small counts (<3)

In [ ]:
# Check minimum number
tab = train.groupby("nace_20_code").orgnr.count()
print(f'Minium number of observations in NACE class: {min(tab)}')

In [ ]:
small_groups = tab[tab<3].index
mask_small = train.nace_20_code.isin(small_groups.tolist())
print(f"The following NACE rev. 2.0 have less than 3 observations and have been removed: {small_groups.tolist()}")
print(f"Number of observations removed due to less than 3 observations: {mask_small.sum()}")

train = train.loc[~mask_small,:].copy()

### Preprocess text

In [ ]:
# Get stopwords
nltk.download('stopwords') # one time to use locally
norwegian_stopwords = set(stopwords.words('norwegian'))|{"as"}

In [ ]:
# Create text variable
train["company_name"] = train["company_name"].fillna("")
train["company_activity"] = train["company_activity"].fillna("")
train["company_text"] = train["company_name"] + " " + train["company_activity"]

In [ ]:
# Preprocess and add in nace 2.1
train["company_text_processed"] = preprocess(train, 
                                             text_variable="company_text", 
                                             stopwords = norwegian_stopwords, 
                                             nace_21_variable = "nace_21_code"
                                            )

### Train model


In [ ]:
# using small set for testing
mask_little = train.nace_20_code.isin(["18.12","25.61","28.21","31.02"])
train = train.loc[mask_little,:].copy()
train.groupby("nace_20_code").count()

In [ ]:
#max_features=5000
max_features=500
C=1

with mlflow.start_run() as run:
    pipe_svc = Pipeline([('countvec', CountVectorizer(max_features=max_features)),
                             ('tf', TfidfTransformer(use_idf=True)),
                             ('clf', CalibratedClassifierCV(estimator=LinearSVC(C=C),
                                                            cv=3, 
                                                            method='sigmoid',
                                                            n_jobs=5),
                             )
                            ])
    pipe_svc.fit(train.company_text_processed , train.nace_20_code)

    result = mlflow.sklearn.log_model(sk_mode=pipe_svc, name="model")
    print("Model URI:", result.model_uri)